# ReviewSense — API Development & Deployment

This notebook demonstrates how the saved ReviewSense machine-learning artifacts
are loaded into a reusable prediction pipeline and exposed through a FastAPI
REST API. It also documents the temporary Google Colab + ngrok integration used
to connect the local frontend during development.

**Important:** ngrok provides a temporary tunnel. This notebook is a development
demonstration, not permanent production hosting.

## 1. Load the saved model artifacts

The inference application needs the trained model, fitted TF-IDF vectorizer,
and the exact custom stopword set used during training. The original training
dataset is not required to serve predictions.

In [ ]:
import os
import re
import joblib

MODEL_PATH = "/content/ReviewSense_SVM_Model.pkl"
VECTORIZER_PATH = "/content/ReviewSense_TFIDF_Vectorizer.pkl"
STOPWORDS_PATH = "/content/ReviewSense_Stopwords.pkl"

loaded_model = joblib.load(MODEL_PATH)
loaded_tfidf = joblib.load(VECTORIZER_PATH)
custom_stop_words = joblib.load(STOPWORDS_PATH)

print("Model artifacts loaded successfully.")

In [ ]:
print("Model type:", type(loaded_model).__name__)
print("SVM C parameter:", loaded_model.C)
print("Vectorizer type:", type(loaded_tfidf).__name__)
print("Vocabulary size:", len(loaded_tfidf.get_feature_names_out()))
print("Custom stopwords:", len(custom_stop_words))

## 2. Build the prediction pipeline

The preprocessing steps must match the training pipeline: remove HTML tags,
convert text to lowercase, remove punctuation while preserving apostrophes,
and remove the selected stopwords. Negation words and contractions are
preserved because they can change sentiment.

In [ ]:
def preprocess_text(text):
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize case
    text = text.lower()

    # Remove punctuation while preserving apostrophes
    text = re.sub(r"[^\w\s']", "", text)

    # Remove the same custom stopwords used during training
    words = text.split()
    filtered_words = [word for word in words if word not in custom_stop_words]

    return " ".join(filtered_words)


def predict_sentiment(review):
    cleaned_review = preprocess_text(review)
    review_vector = loaded_tfidf.transform([cleaned_review])

    prediction = loaded_model.predict(review_vector)[0]
    decision_score = loaded_model.decision_function(review_vector)[0]

    return {
        "sentiment": "Positive" if prediction == 1 else "Negative",
        "decision_score": round(float(decision_score), 4)
    }

In [ ]:
sample_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "The movie was boring, disappointing, and a complete waste of time.",
    "I did not enjoy this movie at all."
]

for review in sample_reviews:
    print("Review:", review)
    print("Result:", predict_sentiment(review))
    print("-" * 60)

### Understanding the decision score

The `decision_score` is the signed distance from the Linear SVM decision
boundary. It is **not a probability** and should not be presented as a
percentage confidence score.

## 3. Prepare the FastAPI backend

FastAPI exposes the prediction pipeline through HTTP endpoints. The root
endpoint provides a simple health message, while `/predict` accepts a review
and returns its predicted sentiment and SVM decision score.

In [ ]:
!pip install -q fastapi uvicorn pyngrok

### Save the preprocessing resource

The API loads the same stopword set as the notebook pipeline.

In [ ]:
joblib.dump(custom_stop_words, STOPWORDS_PATH)
print("Custom stopwords saved:", os.path.exists(STOPWORDS_PATH))

### Create `app.py`

This Colab version uses `/content/` paths. For a local or hosted deployment,
we will replace these with paths relative to the backend project directory and
configure CORS for the actual frontend origin.

In [ ]:
%%writefile app.py

import re
import joblib

from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.middleware.cors import CORSMiddleware

model = joblib.load("/content/ReviewSense_SVM_Model.pkl")
vectorizer = joblib.load("/content/ReviewSense_TFIDF_Vectorizer.pkl")
custom_stop_words = joblib.load("/content/ReviewSense_Stopwords.pkl")

app = FastAPI(
    title="ReviewSense API",
    description="IMDb Movie Review Sentiment Classification API",
    version="1.0.0"
)

# Development setting. Restrict allow_origins before production hosting.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ReviewRequest(BaseModel):
    review: str = Field(min_length=1)

def preprocess_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower()
    text = re.sub(r"[^\w\s']", "", text)
    words = text.split()
    filtered_words = [word for word in words if word not in custom_stop_words]
    return " ".join(filtered_words)

@app.get("/")
def home():
    return {"message": "Welcome to ReviewSense API!"}

@app.post("/predict")
def predict(request: ReviewRequest):
    cleaned_review = preprocess_text(request.review)
    review_vector = vectorizer.transform([cleaned_review])
    prediction = model.predict(review_vector)[0]
    score = model.decision_function(review_vector)[0]
    sentiment = "Positive" if prediction == 1 else "Negative"

    return {
        "review": request.review,
        "sentiment": sentiment,
        "decision_score": round(float(score), 4)
    }


## 4. Run the API server

The server runs on port `8001`, the port used in the successful CORS test during
our development session.

In [ ]:
import subprocess
import sys
import time

server = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "app:app",
    "--host", "0.0.0.0",
    "--port", "8001"
])

time.sleep(4)
print("FastAPI server running:", server.poll() is None)

## 5. Expose the API temporarily with ngrok

The authentication token is read from Google Colab Secrets. Never write the
token itself into the notebook or commit it to GitHub.

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get("NGROK_AUTH_TOKEN")
if not token:
    raise ValueError("Add NGROK_AUTH_TOKEN to Colab Secrets first.")

ngrok.set_auth_token(token)
tunnel = ngrok.connect(8001)
public_api_url = tunnel.public_url

print("Temporary ReviewSense API:", public_api_url)
print("API documentation:", public_api_url + "/docs")

## 6. Test CORS and the prediction endpoint

A browser sends an OPTIONS preflight request before certain cross-origin POST
requests. A successful response should include the appropriate CORS headers.

In [ ]:
import requests

headers = {
    "Origin": "http://127.0.0.1:5500",
    "Access-Control-Request-Method": "POST",
    "Access-Control-Request-Headers": "content-type"
}

preflight = requests.options(
    "http://127.0.0.1:8001/predict",
    headers=headers,
    timeout=15
)

print("Preflight status:", preflight.status_code)
print("Allow-Origin:", preflight.headers.get("access-control-allow-origin"))

In [ ]:
test_response = requests.post(
    public_api_url + "/predict",
    json={"review": "This movie was wonderful and I loved it."},
    timeout=30
)

print("Prediction API status:", test_response.status_code)
print(test_response.json())

## 7. Deployment notes and limitations

- The Colab runtime and ngrok tunnel must remain active for this temporary demo to work.
- The ngrok URL may change when a new tunnel is created.
- The original IMDb dataset is not needed by the inference API; only the saved model artifacts are required.
- The SVM decision score is not a calibrated probability.
- The model was trained on IMDb movie reviews and may not generalize reliably to slang, sarcasm, unrelated statements, or other domains.
- Before production hosting, use portable model paths, pin compatible dependencies, restrict CORS to the frontend domain, and host the frontend and API independently.